# Competition: Race Car

Learn to race a car using the [Gymnasium CarRacing environment](https://gymnasium.farama.org/environments/box2d/car_racing/). Use Deep RL. You can use algorithms from 
* [Stable Baseline 3](https://stable-baselines3.readthedocs.io/en/master/)
* [CleanLR](https://github.com/vwxyzjn/cleanrl)


## Setup

You need:
* Gymnasium (see [Installation Instructions](../common/Setup_Gymnasium.ipynb))
* Patched `gym-classics-1.0.0+internal.rev1` or later (see [Installation instructions](../common/Setup_patched_gym_classics.ipynb))

In [32]:
import numpy as np
np.set_printoptions(precision=2)

In [33]:
import gymnasium as gym
import gym_classics
gym_classics.register('gymnasium')

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/common/gym-classics/gym_classics/__init__.py:87: UserWarning: gym-classics environments were already registered for gymnasium; additional calls to `register()` are ignored.
  warnings.warn("gym-classics environments were already registered for {}; "


In [34]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

## Run The Environment

In [35]:
def run_episode(agent_function, env, max_steps=1000, seed = 1, verbose = False):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (observation and state are the same in fully observable environments)
    observation, info = env.reset(seed=seed)
    
    # run one episode
    G = 0 # undiscounted episode return
    for i in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        # step: execute an action in the environment
        observation_prime, reward, terminated, truncated, info = env.step(action)

        if verbose:
            print (f"Step {i+1}: Obs {np.round(observation, 1)} -> Action {action} - > Reward {np.round(reward,1)}, Obs' {np.round(observation_prime,1)}")

        observation = observation_prime
        G += reward

        if terminated:
            break
  
    if verbose:
        print(f"Episode Return: {G}")
    
    return G

## Simple Agent Function

In [36]:
def speed_kills_driver(observation): 
    """A random agent that selects actions uniformly at random. It ignores the observation.
    
    Actions
    0: do nothing
    1: steer right
    2: steer left
    3: gas
    4: brake
    """
    return np.random.choice([0, 1, 2, 3, 4], p=[0, 0.1, 0.1, 0.8, 0])

## Drive to See the Observation Structure

In [38]:
env = gym.make("CarRacing-v3", continuous=False, render_mode=None)
run_episode(speed_kills_driver, env, seed=0, verbose=True)
env.close()

Step 1: Obs [[[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 ...

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]] -> Action 3 - > Reward 6.2, Obs' [[[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 ...

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]]
Step 2: Obs [[[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
 

## Drive With Video

In [39]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make("CarRacing-v3", continuous=False, render_mode="rgb_array")
env_record = VideoWrapper(env, 'RC', render_fps=30)

G = run_episode(speed_kills_driver, env_record, seed=0)
print(f"Episode Return: {G}")

show(env_record)
env.close()

Videos already exist, I remove them first!


/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/DRL/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode Return: -76.90407523511021
Showing: ./videos/video_RC-episode-0.mp4


## Evaluation

We use as the evaluation score: Average undiscounted returns over 100 random tracks (random seed 0..99).

In [41]:
from tqdm import tqdm

env = gym.make("CarRacing-v3", continuous=False, render_mode=None)

N = 100
score = 0.0
for i in tqdm(range(N), desc="Running episodes"):
    score += run_episode(speed_kills_driver, env, seed=i)

print(f"Average Score over {N} races: {score/N}")

env.close()

Running episodes: 100%|██████████| 100/100 [08:16<00:00,  4.97s/it]

Average Score over 100 races: -50.86803722387677


## Some Hints

I am sure you can do better. Here are some pointers:

* You could use [convolution layers](https://en.wikipedia.org/wiki/Convolutional_layer) to process the input images.
* A single image (observation) does not give you information about the car's speed. This actually means that it has a partial observable state. 
  A popular approach is observation stacking by giving the network the current and several prior observations as the input (adding to the tensor depth). 
  This provides temporal context and let's the network estimate the full state representation (including speed) from the data. 


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)